In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(path + "/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df.describe().loc[["mean","std","min","max"]]

In [ ]:
df.shape

In [ ]:
# Task 1: Write your code here:
ms = df.isna().sum()
ms_cols = ms[ms > 0].index
ms_cols

In [ ]:
df[ms_cols].mean()

In [ ]:
df.fillna(df[ms_cols].mean(),inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
# Task 2: Write your code here:
print(df.duplicated().sum())
df.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:
df.select_dtypes(include=["object"]).columns

In [ ]:
# Task 4: Write your code here:
sc=StandardScaler()
dCols = df.columns.drop("Target") ## desired columns to scale (Not scaleing the Target)
df_scaled = df.copy()
df_scaled[dCols] = sc.fit_transform(df_scaled[dCols])

In [ ]:
df_scaled.head()

In [ ]:
# Task 5: Write your code here:
print(df["Target"].value_counts(normalize=True)*100)
sns.countplot(x=df["Target"])

In [ ]:
# Task 1: Write your code here:
X = df_scaled[dCols].values
y = df_scaled["Target"].values

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
model = CatBoostClassifier()
accuracy_=[]
F1Score=[]

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  # 1. Split data
  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y[train_index], y[test_index]

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  accuracy_.append(accuracy)
  F1Score.append(f1)

In [ ]:
print(f"Accracy mean {np.mean(accuracy_)*100}")
print(f"F1 mean {np.mean(F1Score)*100}")

In [ ]:
# Retrieve CatBoost feature importances and sort them
catboost_importance = list(zip(df.columns, model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(30, 60))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.yticks()
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:
print(f"The most important feature is P_2")

In [ ]:
# Task Bonus: Write your code here: